# Reproducible plots for Bubble Plume Data Repository

This notebook generates selected figures directly from the paper-level master CSV files used by the website.

It reads CSV files from `csv/paper_data_downloads/csv/papers/` and saves PNG figures to `figures/reproducible_plots/`.

## Imports and paths

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "csv" / "paper_data_downloads" / "csv" / "papers"
FIG_DIR = REPO_ROOT / "figures" / "reproducible_plots"
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR, FIG_DIR

## Helper for wide paired-series CSVs

The master CSV files store each curve as adjacent x-y columns. The helper below filters by `plot_id`, drops blank rows, plots each available series, and saves the figure.

In [ ]:
def plot_wide_series(
    df,
    plot_id,
    series_columns,
    title,
    ylabel,
    output_name,
    xlabel=r"$z/D$",
    xscale="log",
    yscale="log",
    legend_loc="best",
):
    """Plot paired x-y series stored in a wide master CSV."""
    plot_df = df[df["plot_id"] == plot_id].copy()
    if plot_df.empty:
        raise ValueError(f"No rows found for plot_id={plot_id}")

    fig, ax = plt.subplots(figsize=(8.0, 5.5), facecolor="white")
    ax.set_facecolor("white")
    plotted = 0

    for label, x_col, y_col, marker in series_columns:
        if x_col not in plot_df.columns or y_col not in plot_df.columns:
            continue

        xy = plot_df[[x_col, y_col]].apply(pd.to_numeric, errors="coerce").dropna()
        if xscale == "log":
            xy = xy[(xy[x_col] > 0) & (xy[y_col] > 0)]
        elif yscale == "log":
            xy = xy[xy[y_col] > 0]

        if xy.empty:
            continue

        xy = xy.sort_values(x_col)
        ax.scatter(
            xy[x_col],
            xy[y_col],
            label=label,
            marker=marker,
            s=46,
            alpha=0.88,
            linewidths=0.45,
            edgecolors="white",
        )
        plotted += 1

    ax.set_xscale(xscale)
    ax.set_yscale(yscale)

    ax.set_title(title, fontsize=14, pad=10)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.tick_params(axis="both", which="major", labelsize=10.5)
    ax.tick_params(axis="both", which="minor", labelsize=9)
    ax.grid(True, which="major", linewidth=0.55, alpha=0.32)
    ax.grid(True, which="minor", linewidth=0.35, alpha=0.16)
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color("#4b5563")
    if plotted:
        ax.legend(
            frameon=True,
            framealpha=0.92,
            facecolor="white",
            edgecolor="#cbd5e1",
            fontsize=8.5,
            loc=legend_loc,
            borderpad=0.45,
            labelspacing=0.35,
            handlelength=1.2,
            handletextpad=0.45,
        )
    fig.tight_layout()

    output_path = FIG_DIR / output_name
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved {output_path}")
    return fig, ax

## Load paper-level master CSV files

In [ ]:
li_2020 = pd.read_csv(DATA_DIR / "li_2020.csv")
wang_2019 = pd.read_csv(DATA_DIR / "wang_lai_socolofsky_2019.csv")

li_2020[["plot_id", "plot_title", "source_figure"]].drop_duplicates(), wang_2019[["plot_id", "plot_title", "source_figure"]].drop_duplicates()

## Li et al. 2020 figures

In [ ]:
li_series = [
    ("Li et al. 2020 air stone", "li2020_air_stone_x", "li2020_air_stone_y", "^"),
    ("Li et al. 2020 single orifice", "li2020_single_orifice_x", "li2020_single_orifice_y", "s"),
    ("Milgram 1983", "milgram83_x", "milgram83_y", "o"),
    ("Fannelop & Sjoen 1980", "fs80_x", "fs80_y", "D"),
    ("Milgram & Van Houten 1982", "mv82_x", "mv82_y", "v"),
]

plot_wide_series(li_2020, "figure_10a", li_series, "Li et al. 2020: plume half-width", r"$b_g/D$", "li_2020_bgD_vs_zhat.png", legend_loc="upper left")
plot_wide_series(li_2020, "figure_10b", li_series, "Li et al. 2020: centerline velocity", r"$W_c/W_s$", "li_2020_WcWs_vs_zhat.png", legend_loc="lower left")
plot_wide_series(li_2020, "figure_11a", li_series, "Li et al. 2020: volume flux", r"$Q/(D^2 W_s)$", "li_2020_Qhat_vs_zhat.png", legend_loc="upper left")
plot_wide_series(li_2020, "figure_11b", li_series, "Li et al. 2020: momentum flux", r"$M/(D^2 W_s^2)$", "li_2020_Mhat_vs_zhat.png", legend_loc="upper left")

## Wang et al. 2019 figures

In [ ]:
wang_series = [
    ("Q1", "q1_x", "q1_y", "o"),
    ("Q2", "q2_x", "q2_y", "s"),
]

plot_wide_series(wang_2019, "figure_8", wang_series, "Wang et al. 2019: mean velocity", r"$U_m/U_s$", "wang_2019_UmUs_vs_zhat.png", yscale="linear", legend_loc="upper right")
plot_wide_series(wang_2019, "figure_10", wang_series, "Wang et al. 2019: entrainment coefficient", r"$\alpha$", "wang_2019_alpha_vs_zhat.png", yscale="linear", legend_loc="upper left")
plot_wide_series(wang_2019, "figure_11", wang_series, "Wang et al. 2019: plume half-width", r"$b_g/D$", "wang_2019_bgD_vs_zhat.png", yscale="linear", legend_loc="upper left")
plot_wide_series(wang_2019, "figure_12a", wang_series, "Wang et al. 2019: volume flux", r"$\hat{Q} = Q/(U_s D^2)$", "wang_2019_Qhat_vs_zhat.png", legend_loc="upper left")
plot_wide_series(wang_2019, "figure_12b", wang_series, "Wang et al. 2019: momentum flux", r"$\hat{M} = M/(U_s^2 D^2)$", "wang_2019_Mhat_vs_zhat.png", legend_loc="upper left")